In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import skew, kurtosis
from statsmodels.stats.outliers_influence import variance_inflation_factor

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, QuantileTransformer
from sklearn.cluster import KMeans
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import RobustScaler
from sklearn.decomposition import PCA
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import silhouette_score, davies_bouldin_score, silhouette_samples, adjusted_rand_score, calinski_harabasz_score
import matplotlib.cm as cm
from sklearn.cluster import DBSCAN
from sklearn.mixture import GaussianMixture

import display

In [ ]:
file_path = 'D:\do_an_2_clustering\data\CC GENERAL.csv'


In [ ]:
def load_raw_data(file_path):
    df = pd.read_csv(file_path)
    print(f"✅ Đã nạp dữ liệu. Kích thước: {df.shape}")
    return df


df = load_raw_data(file_path)


In [ ]:
df_raw = df.copy()

df_raw.info()

In [ ]:
df_raw.describe().T

In [ ]:
df_raw.shape
print(f"data rows : {df_raw.shape[0]}")
print(f"data columns : {df_raw.shape[1]}")

In [ ]:
from pandas.core.arrays import numeric
def diagnostic_statistical_data(df):
    report = pd.DataFrame({
        'Dtype': df.types,
        'Missing_rate (%)': (df.isnull().sum()/len(df)*100).round(2),
        'Unique_values': df.unique(),
        'Min': df.min(numeric_only = True),
        'Max' : df.max(numeric_only = True)
    })

    print( "\n Báo cáo chuẩn đoán dữ liêu ")
    return report


diagnostic_report = diagnostic_statistical_data(df_raw)
display(diagnostic_report)

In [ ]:
print(f" duplicate in dateframe :{df_raw.duplicated().sum()}")

In [ ]:
def initial_cleaning_and_split(df):
    if 'CUST_ID' in df.columns:
        df = df.drop(columns = ['CUST_ID'])

    freq_cols = [col for col in df.columns if 'FREQUENCY'in col]
    initial_rows = len(df)
    for col in freq_cols:
        df = df[df[col] <= 1.0]

    print(f"đã loại bỏ {initial_rows - len(df)} hàng lỗi logic tần suất")

    return df




In [ ]:
def check_skew_data(df):
    
    numeric_cols = df.select_dtypes(include=[np.number]).columns
    skew_series = df[numeric_cols].skew()
    kurtosis_series = df[numeric_cols].kurtosis()
    
    report = pd.DataFrame({
        'column': skew_series.index,
        'skew': skew_series.values,
        'kurtosis': kurtosis_series.values
    })
    
    report['status'] = np.where(report['skew'].abs() > 0.75, 'Non-normal', 'Normal')
    
    for _, row in report.iterrows():
        print(f"Column: {row['column']} has skew: {row['skew']:.4f} ({row['status']})")
        
    return report

skew_report = check_skew_data(df_raw)

In [ ]:
def plot_iqr_boxplots(df,  n_cols=3):
    features_cols = df.select_dtypes(include = [np.number]).columns

    n_features = len(features_cols)
    n_rows = (n_features + n_cols - 1) // n_cols

    fig, axes = plt.subplots(n_rows, n_cols, figsize=(18, 5 * n_rows))
    axes = axes.flatten()

    for i, col in enumerate(features_cols):
        df[col].plot(kind='box', ax=axes[i], patch_artist=True,
                     boxprops=dict(facecolor='lightblue', color='blue'))


    for j in range(i + 1, len(axes)):
        fig.delaxes(axes[j])

    plt.tight_layout()
    plt.show()

def get_iqr_stats(df):
    features_cols = df.select_dtypes(include = [np.number]).columns

    stats = []
    for col in features_cols:
        Q1 = df[col].quantile(0.25)
        Q3 = df[col].quantile(0.75)
        IQR = Q3 - Q1
        lower_bound = Q1 - 1.5 * IQR
        upper_bound = Q3 + 1.5 * IQR

        outliers = df[(df[col] < lower_bound) | (df[col] > upper_bound)].shape[0]
        stats.append({
            'Feature': col,
            'Outliers_Count': outliers,
            'Outliers_Pct': f"{(outliers/len(df)*100):.2f}%"
        })
    return pd.DataFrame(stats)

plot_iqr_boxplots(df_raw)


iqr_report = get_iqr_stats(df_raw)
display(iqr_report)